In [1]:
import os 
from dotenv import load_dotenv
load_dotenv()



True

In [2]:
GROQ_API_KEY= os.getenv("GROQ_API_KEY")
MISTRAL_API_KEY= os.getenv("MISTRAL_API_KEY")
BIFROST_BASE_URL= os.getenv("BIFROST_BASE_URL", "http://localhost:8080")
GROQ_API_KEY= os.getenv("GROQ_API_KEY")
BIFROST_GROQ_VKEY = os.getenv("BIFROST_GROQ_VKEY", "")
BIFROST_MISTRAL_VKEY = os.getenv("BIFROST_MISTRAL_VKEY", "")

print("Keys loaded. Bifrost base URL:", BIFROST_BASE_URL)


Keys loaded. Bifrost base URL: http://localhost:8080


In [3]:
import httpx 

try:
    health = httpx.get(f"{BIFROST_BASE_URL}/health", timeout=3)
    bifrost_ok = health.status_code == 200
except Exception:
    bifrost_ok = False

print("Bifrost:", "Online" if bifrost_ok else "Offline")

if not  bifrost_ok: 
    print("Start it with: docker start bifrost")

Bifrost: Online


In [4]:
TEST_PROMPTS = {
    "simple": "What is the boiling point of water?",
    "reasoning": "Explain the difference between SQL and NoSQL in 3 bullet points.",
    "code": "Write a Python function to check if a string is a palindrome.",
    "duplicate1": "What is Docker used for?",
    "duplicate2": "What is Docker primarily used for?",
    "deepwiki": "What are the core components of LlamaIndex? Use the deepwiki tool to check the run-llama/llama_index repo.",
    "tavily": "Search the web for the latest updates on OpenAI Sora and summarize the top result."
}

In [5]:
from rich.console import Console
from rich.table import Table

GROQ_MODEL = "openai/gpt-oss-120b"
GROQ_MODEL_FALLBACK = "openai/gpt-oss-20b"

MISTRAL_MODEL = "mistral-large-latest"
MISTRAL_MODEL_FALLBACK = "mistral-small-latest"

console = Console()
table = Table(title="🤖 Model Configurations", show_header=True, header_style="bold magenta")

table.add_column("Provider", style="cyan", width=12)
table.add_column("Primary Model", style="green")
table.add_column("Fallback Model", style="yellow")

table.add_row("Groq", GROQ_MODEL, GROQ_MODEL_FALLBACK)
table.add_row("Mistral", MISTRAL_MODEL, MISTRAL_MODEL_FALLBACK)

console.print(table)

                   🤖 Model Configurations                    
┏━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┓
┃ Provider     ┃ Primary Model        ┃ Fallback Model       ┃
┡━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━┩
│ Groq         │ openai/gpt-oss-120b  │ openai/gpt-oss-20b   │
│ Mistral      │ mistral-large-latest │ mistral-small-latest │
└──────────────┴──────────────────────┴──────────────────────┘

## Groq & Mistral

In [6]:
from groq import Groq 

client_groq= Groq(api_key=GROQ_API_KEY)

In [9]:
import time
from rich.console import Console
from rich.panel import Panel
from rich.text import Text

console = Console()

start = time.perf_counter()
response = client_groq.chat.completions.create(
    model=GROQ_MODEL,
    messages=[
        {
            "role": "user",
            "content": TEST_PROMPTS["simple"]
        }
    ]
)
latency_ms = (time.perf_counter() - start) * 1000

# --- Pretty print ---
content = response.choices[0].message.content

console.print(Panel(
    content,
    title=f"[bold cyan]{GROQ_MODEL}[/bold cyan]",
    border_style="cyan"
))

stats = Text()
stats.append(f"⏱  {latency_ms:.0f} ms", style="bold yellow")
stats.append("  |  ", style="dim")
stats.append(f"🔤 {response.usage.total_tokens} tokens", style="bold green")

console.print(stats)


╭────────────────────────────────────────────── openai/gpt-oss-120b ──────────────────────────────────────────────╮
│ The boiling point of pure water is **100 °C (212 °F) at standard atmospheric pressure (1 atm or 101.3 kPa)**.   │
│                                                                                                                 │
│ ### Why it depends on pressure                                                                                  │
│ - **Higher pressure → higher boiling point:** In a pressure cooker (≈15 psi above atmospheric), water boils     │
│ around 120 °C–130 °C, which speeds up cooking.                                                                  │
│ - **Lower pressure → lower boiling point:** At high altitudes the atmospheric pressure is lower, so water boils │
│ at a lower temperature. For example:                                                                            │
│   - Denver, ≈ 5,280 ft (1,600 m): ~95 °C                                                                        │
│   - La Paz, ≈ 12,000 ft (3,650 m): ~85 °C                                                                       │
│                                                                                                                 │
│ ### Factors that can shift the temperature slightly                                                             │
│ | Factor | Effect on boiling point |                                                                            │
│ |--------|--------------------------|                                                                           │
│ | **Impurities / dissolved solutes** (e.g., salt) | Raises it (boiling point elevation) – a few °C for typical  │
│ concentrations |                                                                                                │
│ | **Air bubbles / nucleation sites** | Can cause “superheating” where water exceeds 100 °C before bubbles form, │
│ especially in very smooth containers (microwave heating) |                                                      │
│ | **Container material & shape** | Rough surfaces provide nucleation sites, reducing superheating; smooth glass │
│ can allow it |                                                                                                  │
│                                                                                                                 │
│ ### Quick reference formula                                                                                     │
│ For small pressure changes, the Clausius‑Clapeyron approximation gives a handy estimate:                        │
│                                                                                                                 │
│ [                                                                                                               │
│ \Delta T \approx \frac{T_{\text{b}} \, \Delta P}{\Delta H_{\text{vap}}/R}                                       │
│ \]                                                                                                              │
│                                                                                                                 │
│ where:                                                                                                          │
│                                                                                                                 │
│ - \(T_{\text{b}}\) = 373 K (boiling point at 1 atm)                                                             │
│ - \(\Delta P\) = change in pressure (Pa)                                                                        │
│ - \(\Delta H_{\text{vap}}\) ≈ 40.7 kJ mol⁻¹ (enthalpy of vaporization)                                          │
│ - \(R\) = 8.314 J mol⁻¹ K⁻¹                                                                                     │
│                                                       

⏱  7021 ms  |  🔤 592 tokens

In [10]:
from langchain_mistralai import ChatMistralAI
from langchain_core.messages import HumanMessage
from rich.console import Console
from rich.panel import Panel
from rich.text import Text
import time

console = Console()

client_mistral = ChatMistralAI(model=MISTRAL_MODEL, api_key=MISTRAL_API_KEY)

start = time.perf_counter()
response = client_mistral.invoke([
    HumanMessage(content=TEST_PROMPTS["reasoning"])
])
latency_ms = (time.perf_counter() - start) * 1000

# --- Pretty print ---
content = response.content
total_tokens = response.usage_metadata.get("total_tokens", "?") if response.usage_metadata else "?"

console.print(Panel(
    content,
    title=f"[bold magenta]{MISTRAL_MODEL}[/bold magenta]",
    border_style="magenta"
))

stats = Text()
stats.append(f"⏱  {latency_ms:.0f} ms", style="bold yellow")
stats.append("  |  ", style="dim")
stats.append(f"🔤 {total_tokens} tokens", style="bold green")

console.print(stats)


╭───────────────────────────────────────────── mistral-large-latest ──────────────────────────────────────────────╮
│ Here’s a concise comparison of **SQL** and **NoSQL** in 3 key points:                                           │
│                                                                                                                 │
│ - **Data Model & Structure**                                                                                    │
│   - **SQL**: Uses **structured, tabular schemas** (tables with rows/columns) with predefined relationships      │
│ (e.g., foreign keys). Ideal for complex queries and strict data integrity.                                      │
│   - **NoSQL**: Uses **flexible, schema-less models** like documents (JSON), key-value pairs, graphs, or         │
│ wide-column stores. Adapts easily to unstructured or evolving data.                                             │
│                                                                                                                 │
│ - **Scalability & Performance**                                                                                 │
│   - **SQL**: Scales **vertically** (adding more power to a single server) and excels at **complex               │
│ transactions** (e.g., banking systems). Joins and ACID compliance can slow performance at scale.                │
│   - **NoSQL**: Scales **horizontally** (adding more servers) and is optimized for **high-speed reads/writes**   │
│ (e.g., real-time analytics, social media). Sacrifices some consistency for speed (eventual consistency).        │
│                                                                                                                 │
│ - **Use Cases & Flexibility**                                                                                   │
│   - **SQL**: Best for **structured data with clear relationships** (e.g., ERP, CRM, financial systems) where    │
│ data integrity and complex queries are critical.                                                                │
│   - **NoSQL**: Ideal for **dynamic, large-scale, or unstructured data** (e.g., IoT, user profiles, content      │
│ management) where flexibility, speed, and scalability matter more than rigid schemas.                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

⏱  6190 ms  |  🔤 321 tokens

## Streaming

In [11]:
stream = client_groq.chat.completions.create(
    model=GROQ_MODEL,
    messages=[{"role": "user", "content": TEST_PROMPTS["code"]}],
    stream=True,
)

for chunk in stream:
    delta = chunk.choices[0].delta.content
    if delta:
        print(delta, end ="" , flush=True)
print()

Here’s a compact, well‑documented Python function that tells you whether a given string is a palindrome.  
It works for the usual definition of a palindrome: the sequence reads the same forward and backward **ignoring case and any non‑alphanumeric characters** (so `"A man, a plan, 12321, a canal: Panama"` is considered a palindrome).

```python
import re

def is_palindrome(s: str) -> bool:
    """
    Return True if *s* is a palindrome, False otherwise.

    The check is case‑insensitive and skips any characters that are not
    letters or digits.

    Parameters
    ----------
    s: str
        The string to test.

    Returns
    -------
    bool
        True if *s* reads the same backward as forward (ignoring case
        and non‑alphanumeric characters), False otherwise.
    """
    # Keep only alphanumeric characters and make them lower‑case
    cleaned = re.sub(r'[^A-Za-z0-9]', '', s).lower()

    # Compare the cleaned string with its reverse
    return cleaned == cleaned[::-1]


In [12]:
def naive_manual_fallback(prompt:str) -> str:
    try:
        response = client_groq.chat.completions.create(
            model=GROQ_MODEL,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content
    except Exception as groq_error:
        print("GROQ FAILED , trying mistral as backup:" , groq_error)
        response = client_mistral.chat.complete(
            model=MISTRAL_MODEL,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content


In [13]:
answer = naive_manual_fallback(TEST_PROMPTS["simple"])
print(answer)

The boiling point of water at standard atmospheric pressure (1 atm or 101.3 kPa) is **100 °C** (212 °F). 

*Note:* The exact temperature can vary with altitude and pressure—at higher elevations, where atmospheric pressure is lower, water boils at a lower temperature. For example, at 2,000 m (about 6,560 ft) above sea level, water boils around 93 °C (199 °F).


In [ ]:
## GROQ_MODEL = "NOTHING"

def naive_manual_fallback(prompt:str) -> str:
    try:
        response = client_groq.chat.completions.create(
            model=GROQ_MODEL,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content
    except Exception as groq_error:
        print("GROQ FAILED , trying mistral as backup:" , groq_error)
        response = client_mistral.chat.complete(
            model=MISTRAL_MODEL,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content

answer = naive_manual_fallback(TEST_PROMPTS["simple"])
print(answer)

GROQ FAILED , trying mistral as backup: Error code: 404 - {'error': {'message': 'The model `nothing` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}


AttributeError: 'ChatMistralAI' object has no attribute 'chat'

In [32]:
from openai import OpenAI 

bifrost = OpenAI(
    api_key = "not necessary because we have virtual keys",
    base_url = f"{BIFROST_BASE_URL}/v1"
)

print("Bifrost client ready. Pick a provider via the model string, e.g.:")
print(f"  groq/{GROQ_MODEL}")
print(f"  mistral/{MISTRAL_MODEL}")


Bifrost client ready. Pick a provider via the model string, e.g.:
  groq/openai/gpt-oss-120b
  mistral/mistral-large-latest


In [30]:
import json as _json

def call_bifrost(prompt: str, model: str, fallback_model: str = None):
    headers = {"x-bifrost-fallback-models": fallback_model} if fallback_model else None

    raw = bifrost.chat.completions.with_raw_response.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        extra_headers=headers,
    )
    response = raw.parse()
    provider = _json.loads(raw.text).get("extra_fields", {}).get("provider", "unknown")
    return response.choices[0].message.content, provider


In [10]:
answer , provider = call_bifrost(TEST_PROMPTS["simple"] , f"groq/{GROQ_MODEL}")
print(f"[{provider}] {answer}")

[groq] The boiling point of pure water at **standard atmospheric pressure** (1 atm ≈ 101.3 kPa) is:

- **100 °C** (212 °F)  
- **373.15 K**  

### Why it can change
- **Altitude/Pressure:** At higher elevations the atmospheric pressure is lower, so water boils at a lower temperature (e.g., ~90 °C at 2,000 m or ~68 °C on top of Mt. Everest). Conversely, in a pressure cooker where the pressure is higher, water can boil above 100 °C.
- **Impurities/Dissolved Substances:** Adding solutes (salt, sugar, etc.) raises the boiling point—a phenomenon called *boiling point elevation*.
- **Vacuum:** In a vacuum, water can boil at room temperature or even colder.

### Quick reference table (approximate)

| Approx. Pressure | Boiling Temp (°C) |
|------------------|-------------------|
| 1 atm (sea level) | 100 |
| 0.8 atm (~2,000 ft) | ~96 |
| 0.5 atm (~5,500 ft) | ~84 |
| 0.1 atm (high vacuum) | ~45 |

So, under normal conditions at sea level, water boils at **100 °C** (212 °F).


In [11]:
answer , provider = call_bifrost( "how to learn english , respond in 15 words" , f"mistral/{MISTRAL_MODEL}")
print(f"[{provider}] {answer}")

[mistral] Read daily, listen to podcasts, speak often, watch movies, and practice with native speakers.


## Automatic Provider Fallback

In [12]:
answer , provider = call_bifrost(
    TEST_PROMPTS["duplicate2"] , f"groq/{GROQ_MODEL}",
    fallback_model=f"mistral/{MISTRAL_MODEL}"
)
print(f"[{provider}] {answer}")

[groq] Docker is primarily used for **containerization**—the process of packaging an application together with all of its runtime dependencies (code, libraries, system tools, and settings) into a lightweight, portable **container**. This enables developers and operators to:

- **Run applications consistently** across different environments (development, testing, staging, production) without “it works on my machine” issues.
- **Isolate** applications from each other and from the host system, improving security and stability.
- **Scale** services quickly by deploying multiple container instances (often orchestrated with tools like Docker Swarm or Kubernetes).
- **Simplify deployment** pipelines, CI/CD workflows, and micro‑service architectures by treating containers as immutable artifacts that can be versioned, stored in registries, and reproduced on demand.


In [13]:
# Bad model name (404) — does NOT trigger the Mistral fallback, by design
try:
    answer, provider = call_bifrost(
    TEST_PROMPTS["duplicate2"] , 
    f"groq/fake-model",
    fallback_model=f"mistral/{MISTRAL_MODEL}"
    )
    print(f"[{provider}] {answer}")
except Exception as e:
    print("No fallback triggered, as expected for a 404:", e)


No fallback triggered, as expected for a 404: Error code: 404 - {'is_bifrost_error': False, 'status_code': 404, 'error': {'type': 'invalid_request_error', 'code': 'model_not_found', 'message': 'The model `fake-model` does not exist or you do not have access to it.'}, 'extra_fields': {'routing_info': {'provider': 'groq', 'model': 'fake-model', 'key': 'groq bi'}, 'provider': 'groq', 'original_model_requested': 'fake-model', 'resolved_model_used': 'fake-model', 'request_type': 'chat_completion', 'latency': 367}}


## Load Balancing

In [14]:
from collections import Counter

# 10 - 5 , 5 

def bifrost_key_used(model: str, prompt: str = "Hello") -> str:
    response = httpx.post(
        f"{BIFROST_BASE_URL}/v1/chat/completions",
        json={"model": model, "messages": [{"role": "user", "content": prompt}]},
        timeout=30,
    )
    return response.json().get("extra_fields", {}).get("routing_info", {}).get("key", "unknown")


In [15]:
def stream_bifrost(prompt: str, model: str):
    stream = bifrost.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        stream=True,
    )
    for chunk in stream:
        if chunk.choices[0].delta.content:
            print(chunk.choices[0].delta.content, end="", flush=True)
    print()


In [16]:
stream_bifrost(TEST_PROMPTS["code"], f"groq/{GROQ_MODEL}")

Sure! Below is a simple, well‑documented Python function that checks whether a given string is a palindrome. It ignores case and non‑alphanumeric characters (so `"A man, a plan, a canal: Panama"` is considered a palindrome), but you can easily adjust it if you want a stricter check.

```python
import re

def is_palindrome(s: str) -> bool:
    """
    Return True if `s` is a palindrome, False otherwise.

    The function works in a case‑insensitive manner and ignores all
    characters that are not letters or digits.

    Parameters
    ----------
    s : str
        The string to test.

    Returns
    -------
    bool
        True if `s` reads the same forwards and backwards (ignoring
        case and non‑alphanumeric characters), else False.

    Examples
    --------
    >>> is_palindrome("racecar")
    True
    >>> is_palindrome("RaceCar")
    True
    >>> is_palindrome("A man, a plan, a canal: Panama")
    True
    >>> is_palindrome("hello")
    False
    """
    # Remove everythi

In [17]:
import pprint

response = httpx.get(f"{BIFROST_BASE_URL}/api/logs?limit=5", timeout=5)
logs = response.json() if response.status_code == 200 else {"status": response.status_code}

pprint.pprint(logs)


{'has_logs': True,
 'logs': [{'business_unit_id': None,
           'business_unit_name': None,
           'content_hidden': False,
           'content_summary': 'Write a Python function to check if a string is '
                              'a palindrome. Sure! Below is a simple, '
                              'well‑documented Python function that checks '
                              'whether a given string is a palindrome. It '
                              'ignores case and non‑alphanumeric characters '
                              '(so `"A man, a plan, a canal: Panama"` is '
                              'considered a palindrome), but you can easily '
                              'adjust it if you want a stricter check.\n'
                              '\n'
                              '```python\n'
                              'import re\n'
                              '\n'
                              'def is_palindrome(s: str) -> bool:\n'
                              '

## Virtual Key

In [9]:
from openai import OpenAI

def call_with_virtual_key(virtual_key: str, model: str, prompt: str):
    if not virtual_key:
        print("No virtual key set for this provider — create one at http://localhost:8080 -> Virtual Keys")
        return
    client = OpenAI(api_key=virtual_key, base_url=f"{BIFROST_BASE_URL}/v1")
    try:
        response = client.chat.completions.create(
            model=model,
            messages=[{"role": "user", "content": prompt}],
        )
        print(response.choices[0].message.content)
    except Exception as e:
        print("Call failed — check the virtual key is valid and not over budget:", e)


In [10]:
call_with_virtual_key(BIFROST_GROQ_VKEY, f"groq/{GROQ_MODEL}", TEST_PROMPTS["simple"])

The boiling point of water is **100 °C (212 °F)** when measured at **standard atmospheric pressure** (1 atm ≈ 101.3 kPa) at sea level.  

### Why the temperature can vary
- **Altitude:** At higher elevations the atmospheric pressure is lower, so water boils at a lower temperature (e.g., ~90 °C at 3,000 m / 10,000 ft).
- **Pressure:** Increasing the surrounding pressure raises the boiling point (e.g., in a pressure cooker water can boil around 120 °C or higher).
- **Impurities:** Dissolved substances (salt, sugars, etc.) elevate the boiling point slightly—a phenomenon known as *boiling point elevation*.

### Quick reference table (approximate)

| Elevation (m) | Atmospheric pressure (kPa) | Boiling point (°C) |
|---------------|----------------------------|--------------------|
| 0 (sea level) | 101.3                      | 100.0 |
| 500           | 95.5                       | 96.5 |
| 1,000         | 89.9                       | 96.0 |
| 2,000         | 79.5                       | 93

## MCP with Tools

In [11]:
response = httpx.get(f"{BIFROST_BASE_URL}/api/mcp/clients", timeout=5)
response






<Response [200 OK]>

In [12]:
clients = response.json().get("clients", [])

clients

[{'config': {'client_id': '3d1603af-8760-4656-8013-66be8e9e2ada',
   'name': 'deepwiki',
   'is_code_mode_client': False,
   'connection_type': 'http',
   'connection_string': {'value': 'http************************/mcp',
    'type': 'plain_text'},
   'auth_type': 'none',
   'tools_to_execute': ['*'],
   'tools_to_auto_execute': ['*'],
   'is_ping_available': True,
   'tool_sync_interval': 600000000000,
   'disabled': False,
   'allow_on_all_virtual_keys': False},
  'tools': [{'name': 'ask_question',
    'description': 'Ask any question about a GitHub repository and get an AI-powered, context-grounded response.',
    'parameters': {'type': 'object',
     'properties': {'repoName': {'anyOf': [{'type': 'string'},
        {'items': {'type': 'string'}, 'type': 'array'}],
       'description': 'GitHub repository or list of repositories (max 10) in owner/repo format.'},
      'question': {'description': 'The question to ask about the repository.',
       'type': 'string'}},
     'required': 

In [13]:
registered = {c["config"]["name"] for c in clients}
print("Registered:", registered or "none")

Registered: {'deepwiki', 'tavily'}


In [14]:
def call_with_tools(prompt: str, servers: list[str]) -> dict:
    response = httpx.post(
        f"{BIFROST_BASE_URL}/v1/chat/completions",
        json={
            "model": f"mistral/{MISTRAL_MODEL}",
            "messages": [{"role": "user", "content": prompt}],
            "bifrost": {"mcp": {"enabled": True, "code_mode": True, "servers": servers}},
        },
        timeout=60,
    )
    return response.json()["choices"][0]["message"]

In [15]:
# DeepWiki — ask a question about a GitHub repo
if "deepwiki" in registered:
    message = call_with_tools(TEST_PROMPTS["deepwiki"], ["deepwiki"])
    print(message.get("content") or message.get("tool_calls"))
else:
    print("Skipped — add the deepwiki MCP server first (see Section 2.2)")

Here are the **core components of LlamaIndex**, based on the `run-llama/llama_index` repository:

---

### **1. Core Foundation Layer (`llama-index-core`)**
This layer provides the foundational abstractions and primitives for building LLM applications:
- **Base Abstractions**:
  - **`BaseLLM`**: An interface for language models, supporting chat, completion, and streaming functionalities.
  - **`BaseEmbedding`**: An interface for embedding models (e.g., `VoyageEmbedding`).
  - **`BasePydanticVectorStore`**: An interface for vector databases.
  - **`BaseNode`**: The atomic unit of data, containing text and metadata.
  - **`TransformComponent`**: A base class for node transformations and parsers (e.g., transforming `BaseNode` objects).

- **Orchestration Engine**:
  - **`Workflow`**: An event-driven engine for complex, non-linear LLM logic.

---

### **2. Integration Ecosystem (`llama-index-integrations`)**
This layer provides **300+ plugin implementations** for various components, includ

In [16]:
# Tavily — live web search
if "tavily" in registered:
    message = call_with_tools(TEST_PROMPTS["tavily"], ["tavily"])
    print(message.get("content") or message.get("tool_calls"))
else:
    print("Skipped — add the tavily MCP server first (see Section 2.2)")

Here’s a summary of the latest update on **OpenAI Sora** based on the most recent information:

### **Key Update: Sora Access and Copyright Controversy (2024–2025)**
1. **Temporary Shutdown (November 2024)**
   - OpenAI **paused access to Sora** in late November 2024 after a group of artists **leaked access to the tool in protest**, accusing OpenAI of using them as "PR puppets" without proper credit or compensation. This led to public backlash and a temporary halt in Sora’s availability.
   - Sources: [*Variety*](https://variety.com), [*The Washington Post*](https://www.washingtonpost.com).

2. **Sora 2 Release (September 2025)**
   - OpenAI launched **Sora 2** on **September 30, 2025**, introducing significant updates, including:
     - **Opt-out copyright policy**: Copyright holders must now **explicitly opt out** if they don’t want their content used in Sora’s training data. This sparked criticism from organizations like the **Motion Picture Association (MPA)**, which argued it shif

## Qdrant Cloud

In [20]:
JINA_API_KEY = os.getenv("JINA_API_KEY")
QDRANT_URL = os.getenv("QDRANT_CLUSTER")  
QDRANT_API_KEY = os.getenv("QDRANT_API_KEY")

In [21]:
import os
from qdrant_client import QdrantClient
 
try:
    qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)
    qdrant.get_collections()
except Exception:
    qdrant = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY, port=443)

print("Connected. Existing collections:", [c.name for c in qdrant.get_collections().collections])


Connected. Existing collections: []


## Jına Embeddings

In [22]:
import requests
from langchain_core.embeddings import Embeddings

class JinaEmbeddings(Embeddings):
    def __init__(self, api_key: str, model: str = "jina-embeddings-v4"):
        self.api_key = api_key
        self.model = model

    def _embed(self, texts: list[str], task: str) -> list[list[float]]:
        response = requests.post(
            "https://api.jina.ai/v1/embeddings",
            headers={"Authorization": f"Bearer {self.api_key}"},
            json={"model": self.model, "input": texts, "task": task},
            timeout=30,
        )
        response.raise_for_status()
        return [item["embedding"] for item in response.json()["data"]]

    def embed_documents(self, texts: list[str]) -> list[list[float]]:
        return self._embed(texts, task="retrieval.passage")

    def embed_query(self, text: str) -> list[float]:
        return self._embed([text], task="retrieval.query")[0]

jina = JinaEmbeddings(api_key=JINA_API_KEY)


In [23]:
JINA_DIM = len(jina.embed_query("The Crow"))
print("Embeddings length" , JINA_DIM)

Embeddings length 2048


## Knowledge Base with Qdrant Cloud

In [24]:
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

COLLECTION = "bifrost_rag_demo"

DOCS = [
    "Hermes is a Rust-based, cloud-native ML toolkit made by Cohere AI, deployed under BSD 1.0.",
    "Hermes handles automatic failover across endpoints, launched on 5xx issues, 503 rate limits, and 403 auth blockages.",
    "Hermes provides a single Claude-compatible /v2/text/completions endpoint; the pipeline is configured via a \"pipeline/model\" string.",
    "Nova hosts SLMs on custom TPU hardware, granting ultra low-latency execution with an open SDK plan.",
    "Anthropic AI offers Le Dashboard with a free tier for models like claude-large-latest and claude-small-latest.",
    "Milvus Cloud is a managed vector database used here for NLP retrieval, reached over a URI + SDK key.",
    "Nomic AI provides an embeddings API (nomic-embeddings-v2) with task-specific adapters for retrieval and classification.",
]

if qdrant.collection_exists(COLLECTION):
    qdrant.delete_collection(COLLECTION)
    
qdrant.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=JINA_DIM, distance=Distance.COSINE),
)   

vectorstore = QdrantVectorStore(
    client = qdrant , 
    collection_name=COLLECTION,
    embedding=jina
)

print("COLLECTION_READY")


COLLECTION_READY


In [25]:
print("COLLECTION_READY" , COLLECTION)

COLLECTION_READY bifrost_rag_demo


In [26]:
vectorstore.add_texts(DOCS)

['c1458421719841fabfc8690328e8c294',
 '917dec01aa334e6c99d88f9a3387a24f',
 '45e781e00b5d4cc0aa55097e82ddb2a8',
 '336ef0d152a145a6a3e5bae81d95932b',
 '5bcfc4b6d3e64d928a8b7b9f395494fa',
 '4afbad0785434fd698492900966b090e',
 'e204b935b7a1474885e9646d01131aa4']

## RAG Pipeline

In [27]:
def rag_query(question: str, model: str = f"groq/{GROQ_MODEL}", top_k: int = 3) -> dict:
    hits = vectorstore.similarity_search_with_score(question, k=top_k)

    context = "\n".join(f"- {doc.page_content}" for doc, _score in hits)
    prompt = (
    "You are a Hermes agent. Answer the question using ONLY the context provided below. "
    "If the answer is not present in the context, explicitly state that the provided information is insufficient.\n\n"
    f"Context:\n{context}\n\nQuestion: {question}"
)

    answer, provider = call_bifrost(prompt, model)
    return {"retrieved": hits, "answer": answer, "provider": provider}


In [34]:
result = rag_query("Under what conditions does Hermes trigger automatic failover?")

print("Retrieved context:")
for doc, score in result["retrieved"]:
    print(f"  [{round(score, 3)}] {doc.page_content}") 

Retrieved context:
  [0.682] Hermes handles automatic failover across endpoints, launched on 5xx issues, 503 rate limits, and 403 auth blockages.
  [0.545] Hermes is a Rust-based, cloud-native ML toolkit made by Cohere AI, deployed under BSD 1.0.
  [0.463] Hermes provides a single Claude-compatible /v2/text/completions endpoint; the pipeline is configured via a "pipeline/model" string.


In [35]:
result = rag_query("What programming language is Nova built with?")


print("Retrieved context:")
for doc, score in result["retrieved"]:
    print(f"  [{round(score, 3)}] {doc.page_content}")

Retrieved context:
  [0.68] Nova hosts SLMs on custom TPU hardware, granting ultra low-latency execution with an open SDK plan.
  [0.502] Hermes is a Rust-based, cloud-native ML toolkit made by Cohere AI, deployed under BSD 1.0.
  [0.479] Milvus Cloud is a managed vector database used here for NLP retrieval, reached over a URI + SDK key.


In [37]:

context = "\n".join([doc.page_content for doc, _ in result["retrieved"]])
question = "What programming language is Nova built with?"


prompt = (
    "You are a Hermes agent. Answer the question using ONLY the context provided below. "
    "If the answer is not present in the context, explicitly state that the provided information is insufficient.\n\n"
    f"Context:\n{context}\n\n"
    f"Question: {question}"
)


response = client_groq.invoke(prompt)

print("--- LLM Response ---")
print(response.content if hasattr(response, "content") else response)

AttributeError: 'Groq' object has no attribute 'invoke'